<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 29 · Derivatives Valuation

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the code examples from the chapter in a Colab-ready
format so that you can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the book text for detailed explanations and context.


In [ ]:
import sys
import math
import datetime as dt
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

# Add the code directory to sys.path so dxlib can be imported
CODE_DIR = (Path("..") / "code").resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from typing import Any, Protocol
from dxlib import *

class Pricer(Protocol):
    def value(self, spot: float) -> tuple[float, float]: ...
    
snapshot_path = Path("..") / "data" / "spx_options_snapshot.csv"
snap = load_spx_snapshot(snapshot_path) if snapshot_path.exists() else None
if snap is not None:
    short_expiry = dt.date(2026, 3, 20)
    long_expiries = [dt.date(2026, 6, 18), dt.date(2026, 12, 18)]
    short_surface_df = select_small_surface(snap, [short_expiry], rate=0.03)
    long_surface_df = select_small_surface(snap, long_expiries, rate=0.03)
    surface_df = long_surface_df
    paths = 15_000
    steps_per_year = 320
    rate = 0.03
    h_params = HestonParams(
        kappa=2.6,
        theta=0.046,
        vol_of_vol=0.9,
        rho=-0.67,
        v0=0.018,
    )
    params = h_params
    local_map = {
        dt.date(2026, 6, 18): (0.0606, 0.0128, -0.732),
        dt.date(2026, 12, 18): (0.0484, 0.0149, -0.763),
    }
    seed = 11

## From Paths to Values

Execute the code examples below.


## Discounting on a Year-Fraction Time Grid

Execute the code examples below.


In [ ]:
class DiscountingModel(Protocol):
    def discount_factor(self, ttm: float) -> float: ...

In [ ]:
@dataclass(frozen=True, slots=True)
class FlatDiscounting:
    rate: float

    def discount_factor(self, ttm: float) -> float:
        return float(math.exp(-self.rate * float(ttm)))

## Payoffs as Small, Explicit Classes

Execute the code examples below.


In [ ]:
@dataclass(frozen=True, slots=True)
class EuropeanPut:
    strike: float

    def __call__(self, spot):
        return np.maximum(self.strike - spot, 0.0)

In [ ]:
@dataclass(frozen=True, slots=True)
class AmericanPut:
    strike: float

    def intrinsic_value(self, spot):
        return np.maximum(self.strike - spot, 0.0)

## European Valuation by Monte Carlo

Execute the code examples below.


In [ ]:
@dataclass(frozen=True, slots=True)
class EuropeanMCPricer:
    process: PathSimulator
    payoff: TerminalPayoff
    discounting: DiscountingModel
    maturity: float
    steps: int
    paths: int

    def value(self, spot):
        time_grid = build_time_grid(self.maturity, self.steps)
        paths = self.process.simulate_paths(spot, time_grid, self.paths)
        payoff = self.payoff(paths[:, -1])
        pv = self.discounting.discount_factor(self.maturity) * payoff
        price = float(np.mean(pv))
        stderr = float(np.std(pv, ddof=1) / math.sqrt(self.paths))
        return price, stderr

## Why American Options Are Different

Execute the code examples below.


## The Longstaff-Schwartz (LSM) Algorithm

Execute the code examples below.


## Implementing LSM as a Class

Execute the code examples below.


## An American Put on an Equity Index

Execute the code examples below.


## Monte Carlo Diagnostics (Figures)

Execute the code examples below.


## Interactive Session: Pricing a European and an American Put

Execute the code examples below.


In [ ]:
import numpy as np

from dxlib import (
    AmericanPut,
    AmericanPutLSM,
    EuropeanPut,
    EuropeanMCPricer,
    FlatDiscounting,
    GeometricBrownianMotion,
)

In [ ]:
s0 = 36.0

k = 40.0

r = 0.06

sigma = 0.2

ttm = 1.0

steps = 12

disc = FlatDiscounting(rate=r)

gbm_q = GeometricBrownianMotion(
    drift=r,
    volatility=sigma,
    seed=7,
)

In [ ]:
euro_put = EuropeanPut(strike=k)

euro_pricer = EuropeanMCPricer(
    process=gbm_q,
    payoff=euro_put,
    discounting=disc,
    maturity=ttm,
    steps=steps,
    paths=200_000,
)

euro_price, euro_err = euro_pricer.value(spot=s0)

round(float(euro_price), 4), round(float(euro_err), 4)

In [ ]:
am_put = AmericanPut(strike=k)

lsm = AmericanPutLSM(
    process=gbm_q,
    payoff=am_put,
    discounting=disc,
    maturity=ttm,
    steps=steps,
    paths=200_000,
    basis_degree=2,
)

res = lsm.value(spot=s0)

round(float(res["price"]), 4), round(float(res["stderr"]), 4)

## Where We Are Heading Next

Execute the code examples below.


## Appendix: `dxlib` Valuation Source Code

Execute the code examples below.


## `code/dxlib/discounting.py`

Execute the code examples below.


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 29 - Derivatives Valuation.

Deterministic discounting building blocks for Monte Carlo valuation.

(c) Dr. Yves J. Hilpisch
AI-supported by GPT 5.x
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""


import math
from dataclasses import dataclass
from typing import Protocol

import numpy as np
from numpy.typing import NDArray

__all__ = ["DiscountingModel", "FlatDiscounting", "discount_factors"]

FloatArray = NDArray[np.float64]


class DiscountingModel(Protocol):
    """
    Discounting model interface used in Monte Carlo valuation.

    The time argument is a year fraction (time-to-maturity) measured from 0.
    """

    def discount_factor(self, ttm: float) -> float: ...


@dataclass(frozen=True, slots=True)
class FlatDiscounting:
    """
    Constant continuously compounded short-rate discounting.
    """

    rate: float

    def discount_factor(self, ttm: float) -> float:
        if ttm < 0:
            raise ValueError("ttm must be non-negative")
        return float(math.exp(-self.rate * float(ttm)))


def discount_factors(model: DiscountingModel, times: FloatArray) -> FloatArray:
    """
    Vectorized discount factors for a NumPy array of year fractions.
    """

    if times.ndim != 1:
        raise ValueError("times must be one-dimensional")
    if np.any(times < 0):
        raise ValueError("times must be non-negative")

    out = np.empty_like(times, dtype=float)
    for idx, ttm in enumerate(times):
        out[idx] = model.discount_factor(float(ttm))
    return out

## `code/dxlib/payoffs.py`

Execute the code examples below.


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 29 - Derivatives Valuation.

Payoff functions used in Part VI.

(c) Dr. Yves J. Hilpisch
AI-supported by GPT 5.x
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""


from dataclasses import dataclass
from typing import Protocol

import numpy as np
from numpy.typing import NDArray

__all__ = ["TerminalPayoff", "EuropeanCall", "EuropeanPut", "AmericanPut"]

FloatArray = NDArray[np.float64]


class TerminalPayoff(Protocol):
    """
    Payoff depending only on the terminal risk factor value.
    """

    def __call__(self, spot: FloatArray) -> FloatArray: ...


@dataclass(frozen=True, slots=True)
class EuropeanCall:
    strike: float

    def __post_init__(self) -> None:
        if self.strike <= 0:
            raise ValueError("strike must be positive")

    def __call__(self, spot: FloatArray) -> FloatArray:
        return np.maximum(spot - self.strike, 0.0)


@dataclass(frozen=True, slots=True)
class EuropeanPut:
    strike: float

    def __post_init__(self) -> None:
        if self.strike <= 0:
            raise ValueError("strike must be positive")

    def __call__(self, spot: FloatArray) -> FloatArray:
        return np.maximum(self.strike - spot, 0.0)


@dataclass(frozen=True, slots=True)
class AmericanPut:
    strike: float

    def __post_init__(self) -> None:
        if self.strike <= 0:
            raise ValueError("strike must be positive")

    def intrinsic_value(self, spot: FloatArray) -> FloatArray:
        return np.maximum(self.strike - spot, 0.0)

    def __call__(self, spot: FloatArray) -> FloatArray:
        return self.intrinsic_value(spot)

## `code/dxlib/valuation.py`

Execute the code examples below.


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 29 - Derivatives Valuation.

Monte Carlo valuation helpers.

(c) Dr. Yves J. Hilpisch
AI-supported by GPT 5.x
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""


import math
from dataclasses import dataclass

import numpy as np

from dxlib.discounting import DiscountingModel
from dxlib.payoffs import TerminalPayoff
from dxlib.processes import PathSimulator, build_time_grid

__all__ = ["EuropeanMCPricer"]


@dataclass(frozen=True, slots=True)
class EuropeanMCPricer:
    """
    Terminal-payoff Monte Carlo valuation.
    """

    process: PathSimulator
    payoff: TerminalPayoff
    discounting: DiscountingModel
    maturity: float
    steps: int
    paths: int

    def __post_init__(self) -> None:
        if self.maturity <= 0:
            raise ValueError("maturity must be positive")
        if self.steps <= 0:
            raise ValueError("steps must be positive")
        if self.paths <= 0:
            raise ValueError("paths must be positive")

    def value(self, spot: float) -> tuple[float, float]:
        time_grid = build_time_grid(self.maturity, self.steps)
        paths = self.process.simulate_paths(
            spot=spot,
            time_grid=time_grid,
            paths=self.paths,
        )
        payoff = self.payoff(paths[:, -1])
        df = self.discounting.discount_factor(self.maturity)
        pv = df * payoff
        price = float(np.mean(pv))
        stderr = float(np.std(pv, ddof=1) / math.sqrt(self.paths))
        return price, stderr

## `code/dxlib/lsm.py`

Execute the code examples below.


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 29 - Derivatives Valuation.

Least-squares Monte Carlo (Longstaff-Schwartz) valuation for American options.

(c) Dr. Yves J. Hilpisch
AI-supported by GPT 5.x
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""


import math
from dataclasses import dataclass

import numpy as np
from numpy.typing import NDArray

from dxlib.discounting import DiscountingModel, discount_factors
from dxlib.payoffs import AmericanPut
from dxlib.processes import PathSimulator, build_time_grid

__all__ = ["AmericanPutLSM", "lsm_american_put_from_paths"]

FloatArray = NDArray[np.float64]


def _poly_basis(x: FloatArray, degree: int) -> FloatArray:
    if degree < 0:
        raise ValueError("degree must be non-negative")
    columns = [np.ones_like(x)]
    for pow_ in range(1, degree + 1):
        columns.append(x**pow_)
    return np.column_stack(columns).astype(float, copy=False)


def lsm_american_put_from_paths(
    spot_paths: FloatArray,
    *,
    strike: float,
    discounting: DiscountingModel,
    time_grid: FloatArray,
    basis_degree: int = 2,
    min_itm: int = 200,
) -> dict[str, object]:
    """
    LSM valuation for an American put using pre-simulated spot paths.
    """

    if spot_paths.ndim != 2:
        raise ValueError("spot_paths must be two-dimensional")
    if time_grid.ndim != 1:
        raise ValueError("time_grid must be one-dimensional")
    if spot_paths.shape[1] != time_grid.size:
        raise ValueError("spot_paths and time_grid must align")
    if strike <= 0:
        raise ValueError("strike must be positive")
    if basis_degree < 1:
        raise ValueError("basis_degree must be at least 1")
    if min_itm < 10:
        raise ValueError("min_itm must be at least 10")

    payoff = AmericanPut(strike=float(strike))
    intrinsic = payoff.intrinsic_value(spot_paths)
    cashflow = intrinsic[:, -1].astype(float, copy=True)

    df_grid = discount_factors(discounting, time_grid)
    df_step = df_grid[1:] / df_grid[:-1]

    steps = int(time_grid.size - 1)
    boundary = np.full(time_grid.size, np.nan, dtype=float)
    exercise_index = np.full(spot_paths.shape[0], steps, dtype=int)

    for step in range(steps - 1, 0, -1):
        cashflow *= float(df_step[step])

        spot_t = spot_paths[:, step]
        intrinsic_t = intrinsic[:, step]
        itm = intrinsic_t > 0.0
        if np.count_nonzero(itm) < min_itm:
            continue

        x = (spot_t[itm] / float(strike)).astype(float, copy=False)
        y = cashflow[itm]
        x_mat = _poly_basis(x, degree=basis_degree)
        beta, *_ = np.linalg.lstsq(x_mat, y, rcond=None)
        continuation = x_mat @ beta

        exercise = intrinsic_t[itm] > continuation
        if np.any(exercise):
            exercised_spots = spot_t[itm][exercise]
            boundary[step] = float(np.max(exercised_spots))

        new_cashflow = cashflow[itm].copy()
        new_cashflow[exercise] = intrinsic_t[itm][exercise]
        cashflow[itm] = new_cashflow

        exercise_paths = np.flatnonzero(itm)[exercise]
        exercise_index[exercise_paths] = step

    pv0 = cashflow * float(df_step[0])
    price = float(np.mean(pv0))
    stderr = float(np.std(pv0, ddof=1) / math.sqrt(spot_paths.shape[0]))

    return {
        "price": price,
        "stderr": stderr,
        "time_grid": time_grid,
        "exercise_boundary": boundary,
        "exercise_index": exercise_index,
    }


@dataclass(frozen=True, slots=True)
class AmericanPutLSM:
    """
    Least-squares Monte Carlo (LSM) valuation for an American put option.

    The implementation follows Longstaff & Schwartz (2001) and uses polynomial
    basis functions in the normalized state variable latexmath:[S_t / K].
    """

    process: PathSimulator
    payoff: AmericanPut
    discounting: DiscountingModel
    maturity: float
    steps: int
    paths: int
    basis_degree: int = 2
    min_itm: int = 200

    def __post_init__(self) -> None:
        if self.maturity <= 0:
            raise ValueError("maturity must be positive")
        if self.steps <= 1:
            raise ValueError("steps must be at least 2")
        if self.paths <= 0:
            raise ValueError("paths must be positive")
        if self.basis_degree < 1:
            raise ValueError("basis_degree must be at least 1")
        if self.min_itm < 10:
            raise ValueError("min_itm must be at least 10")

    def value(self, spot: float) -> dict[str, object]:
        strike = self.payoff.strike
        time_grid = build_time_grid(self.maturity, self.steps)
        spot_paths = self.process.simulate_paths(
            spot=spot,
            time_grid=time_grid,
            paths=self.paths,
        )

        intrinsic = self.payoff.intrinsic_value(spot_paths)
        cashflow = intrinsic[:, -1].astype(float, copy=True)

        df_grid = discount_factors(self.discounting, time_grid)
        df_step = df_grid[1:] / df_grid[:-1]

        boundary = np.full(time_grid.size, np.nan, dtype=float)
        exercise_index = np.full(self.paths, self.steps, dtype=int)

        for step in range(self.steps - 1, 0, -1):
            cashflow *= float(df_step[step])

            spot_t = spot_paths[:, step]
            intrinsic_t = intrinsic[:, step]
            itm = intrinsic_t > 0.0
            if np.count_nonzero(itm) < self.min_itm:
                continue

            x = (spot_t[itm] / strike).astype(float, copy=False)
            y = cashflow[itm]
            x_mat = _poly_basis(x, degree=self.basis_degree)
            beta, *_ = np.linalg.lstsq(x_mat, y, rcond=None)
            continuation = x_mat @ beta

            exercise = intrinsic_t[itm] > continuation
            if np.any(exercise):
                exercised_spots = spot_t[itm][exercise]
                boundary[step] = float(np.max(exercised_spots))

            new_cashflow = cashflow[itm].copy()
            new_cashflow[exercise] = intrinsic_t[itm][exercise]
            cashflow[itm] = new_cashflow

            exercise_paths = np.flatnonzero(itm)[exercise]
            exercise_index[exercise_paths] = step

        pv0 = cashflow * float(df_step[0])
        price = float(np.mean(pv0))
        stderr = float(np.std(pv0, ddof=1) / math.sqrt(self.paths))

        out: dict[str, object] = {
            "price": price,
            "stderr": stderr,
            "time_grid": time_grid,
            "exercise_boundary": boundary,
            "exercise_index": exercise_index,
        }
        return out

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
